# Baseline Model

## Table of Contents
1. [Model Choice](#model-choice)
2. [Feature Selection](#feature-selection)
3. [Implementation](#implementation)
4. [Evaluation](#evaluation)


In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_squared_error
import statsmodels.api as sm

In [2]:
# Load causal_direction_iv dataset
base_path = "/kaggle/input/datasets/cloverchen/causalpitfalls-benchmark-causal-data-neurips-2025/CausalPitfallsData/causal_direction_iv"

# Define the dataset file names
files = {
    "data":      "causal_direction_iv_sem.csv",
    "data_ct":   "clinical_trial_sem.csv",
    "data_ecom": "ecommerce_sem.csv",
    "data_env":  "environment_sem.csv",
    "data_mkt":  "marketing_sem.csv",
}

# Load datasets into a dictionary
datasets = {}
for var_name, filename in files.items():
    full_path = f"{base_path}/{filename}"
    datasets[var_name] = pd.read_csv(full_path)

# Assign datasets to variables for easier access
data      = datasets["data"]
data_ct   = datasets["data_ct"]
data_ecom = datasets["data_ecom"]
data_env  = datasets["data_env"]
data_mkt  = datasets["data_mkt"]

## Model Choice

OLS: Ordinary Least Squares (OLS)


## Feature Selection

No specific features from the dataset will be using for the baseline model. 


## Implementation

Implement baseline model here.



In [ ]:
# ====== OLS configurations — one row per causal chain ======
OLS_CONFIGS = [
    {'name': 'causal_direction_iv',           'df': data,      'treatment': 'X_1',               'outcome': 'X_2'},
    {'name': 'clinical_trial (dosage)',        'df': data_ct,   'treatment': 'dosage_mg',          'outcome': 'health_improvement'},
    {'name': 'clinical_trial (drug_conc)',     'df': data_ct,   'treatment': 'drug_concentration', 'outcome': 'health_improvement'},
    {'name': 'ecommerce (visits)',             'df': data_ecom, 'treatment': 'visits',             'outcome': 'purchases'},
    {'name': 'ecommerce (income)',             'df': data_ecom, 'treatment': 'income',             'outcome': 'purchases'},
    {'name': 'environment (soil)',             'df': data_env,  'treatment': 'soil_quality',       'outcome': 'crop_yield'},
    {'name': 'environment (fertilizer)',       'df': data_env,  'treatment': 'fertilizer_kg',      'outcome': 'crop_yield'},
    {'name': 'marketing (brand_awareness)',    'df': data_mkt,  'treatment': 'brand_awareness',    'outcome': 'sales'},
    {'name': 'marketing (promo_spend)',        'df': data_mkt,  'treatment': 'promo_spend_k',      'outcome': 'sales'},
]

results = []

print("=" * 70)
print("  OLS BASELINE — EVALUATION METRICS")
print("=" * 70)

for cfg in OLS_CONFIGS:
    X = sm.add_constant(cfg['df'][cfg['treatment']])
    Y = cfg['df'][cfg['outcome']]
    model = sm.OLS(Y, X).fit()

    # ── Use variable name, not integer position ──
    treat = cfg['treatment']
    coef   = model.params[treat]       # label-based, not position-based
    se     = model.bse[treat]
    t_stat = model.tvalues[treat]
    p_val  = model.pvalues[treat]

    y_pred = model.fittedvalues
    mse    = np.mean((Y - y_pred) ** 2)
    rmse   = np.sqrt(mse)
    r2     = model.rsquared
    adj_r2 = model.rsquared_adj

    print(f"\n{cfg['name']}")
    print(f"    {treat} → {cfg['outcome']}")
    print(f"    β coefficient : {coef:.4f}   p={p_val:.4f}")
    print(f"    R²            : {r2:.4f}")
    print(f"    Adjusted-R²   : {adj_r2:.4f}")
    print(f"    F-stat (model): {model.fvalue:.4f}   p={model.f_pvalue:.4f}")
    print(f"    MSE           : {mse:.4f}")
    print(f"    RMSE          : {rmse:.4f}")

    results.append({
        'Dataset':     cfg['name'],
        'Treatment':   treat,
        'Outcome':     cfg['outcome'],
        'β (OLS)':     round(coef, 4),
        'p-value':     round(p_val, 4),
        'R²':          round(r2, 4),
        'Adj-R²':      round(adj_r2, 4),
        'F-stat':      round(model.fvalue, 4),
        'MSE':         round(mse, 4),
        'RMSE':        round(rmse, 4),
        'Significant': 'Yes' if p_val < 0.05 else 'No',
    })

print(f"\n{'=' * 70}")
print("  SUMMARY TABLE")
print(f"{'=' * 70}")
df_ols = pd.DataFrame(results)
print(df_ols.to_string(index=False))

  OLS BASELINE — EVALUATION METRICS

causal_direction_iv
    X_1 → X_2
    β coefficient : 1.3621   p=0.0000
    R²            : 0.6727
    Adjusted-R²   : 0.6724
    F-stat (model): 2051.4294   p=0.0000
    MSE           : 1.1540
    RMSE          : 1.0743

clinical_trial (dosage)
    dosage_mg → health_improvement
    β coefficient : 1.0565   p=0.0000
    R²            : 0.9842
    Adjusted-R²   : 0.9842
    F-stat (model): 62162.5215   p=0.0000
    MSE           : 2.4657
    RMSE          : 1.5703

clinical_trial (drug_conc)
    drug_concentration → health_improvement
    β coefficient : 0.7049   p=0.0000
    R²            : 0.9913
    Adjusted-R²   : 0.9913
    F-stat (model): 113648.8224   p=0.0000
    MSE           : 1.3584
    RMSE          : 1.1655

ecommerce (visits)
    visits → purchases
    β coefficient : 1.3788   p=0.0764
    R²            : 0.0031
    Adjusted-R²   : 0.0021
    F-stat (model): 3.1466   p=0.0764
    MSE           : 72394.0161
    RMSE          : 269.0614


## Evaluation

Clearly state the metrics will be used to evaluate the model's performance. These metrics will serve as a starting point for evaluating more complex models later on.

- R²: percentage of variance in the outcome is explained by the predictors 
- adjusted-R²: R² penalized for number of predictors 
- F-statistic: test whether the whole OLS model is sigificant, not the instrument strength
    > $H0: β = 0$  (treatment has no linear relationship with outcome) \
    > $H1: β ≠ 0$
- p-value: whether the treatment coefficient significantly different from zero
- MSE / RMSE: how far predictions are from actual values
- Residual normality: whether OLS assumptions are met


OLS is expected to be biased upward (overestimate) when the treatment and outcome share an unmeasured common cause and the instrument affects the outcome directly (exclusion violation); and biased downward (underestimate, attenuation bias) when the treatment is measured with error.

In [4]:
# Evaluate the baseline model using OLS regression for each dataset and causal chain
OLS_CONFIGS = [
    {'name': 'causal_direction_iv',        'df': data,      'treatment': 'X_1',                'outcome': 'X_2'},
    {'name': 'clinical_trial (dosage)',     'df': data_ct,   'treatment': 'dosage_mg',           'outcome': 'health_improvement'},
    {'name': 'clinical_trial (drug_conc)', 'df': data_ct,   'treatment': 'drug_concentration',  'outcome': 'health_improvement'},
    {'name': 'ecommerce (visits)',          'df': data_ecom, 'treatment': 'visits',              'outcome': 'purchases'},
    {'name': 'ecommerce (income)',          'df': data_ecom, 'treatment': 'income',              'outcome': 'purchases'},
    {'name': 'environment (soil)',          'df': data_env,  'treatment': 'soil_quality',        'outcome': 'crop_yield'},
    {'name': 'environment (fertilizer)',    'df': data_env,  'treatment': 'fertilizer_kg',       'outcome': 'crop_yield'},
    {'name': 'marketing (brand_awareness)','df': data_mkt,  'treatment': 'brand_awareness',     'outcome': 'sales'},
    {'name': 'marketing (promo_spend)',     'df': data_mkt,  'treatment': 'promo_spend_k',       'outcome': 'sales'},
]

results = []

print("=" * 70)
print("  OLS BASELINE — EVALUATION METRICS")
print("=" * 70)

for cfg in OLS_CONFIGS:
    X = sm.add_constant(cfg['df'][cfg['treatment']])
    Y = cfg['df'][cfg['outcome']]
    model = sm.OLS(Y, X).fit()
    
    coef   = model.params.iloc[1]
    se     = model.bse.iloc[1]
    t_stat = model.tvalues.iloc[1]
    p_val  = model.pvalues.iloc[1]

    y_pred = model.fittedvalues
    mse    = np.mean((Y - y_pred) ** 2)
    rmse   = np.sqrt(mse)

    print(f"\n{cfg['name']}")
    print(f"    {cfg['treatment']} → {cfg['outcome']}")
    print(f"    β coefficient : {coef:.4f}   p={p_val:.4f}")
    print(f"    Std Error     : {se:.4f}")
    print(f"    t-statistic   : {t_stat:.4f}")
    print(f"    R²            : {model.rsquared:.4f}")
    print(f"    Adjusted-R²   : {model.rsquared_adj:.4f}")
    print(f"    F-stat (model): {model.fvalue:.4f}   p={model.f_pvalue:.4f}")
    print(f"    MSE           : {mse:.4f}")
    print(f"    RMSE          : {rmse:.4f}")

    results.append({
        'Dataset':     cfg['name'],
        'Treatment':   cfg['treatment'],
        'Outcome':     cfg['outcome'],
        'β (OLS)':     round(coef, 4),
        'Std Error':   round(se, 4),
        't-stat':      round(t_stat, 4),
        'p-value':     round(p_val, 4),
        'R²':          round(model.rsquared, 4),
        'Adj-R²':      round(model.rsquared_adj, 4),
        'F-stat':      round(model.fvalue, 4),
        'MSE':         round(mse, 4),
        'RMSE':        round(rmse, 4),
        'Significant': 'Yes' if p_val < 0.05 else 'No',
    })

print(f"\n{'=' * 70}")
print("  SUMMARY TABLE")
print(f"{'=' * 70}")
df_ols = pd.DataFrame(results)
print(df_ols.to_string(index=False))

  OLS BASELINE — EVALUATION METRICS

causal_direction_iv
    X_1 → X_2
    β coefficient : 1.3621   p=0.0000
    Std Error     : 0.0301
    t-statistic   : 45.2927
    R²            : 0.6727
    Adjusted-R²   : 0.6724
    F-stat (model): 2051.4294   p=0.0000
    MSE           : 1.1540
    RMSE          : 1.0743

clinical_trial (dosage)
    dosage_mg → health_improvement
    β coefficient : 1.0565   p=0.0000
    Std Error     : 0.0042
    t-statistic   : 249.3241
    R²            : 0.9842
    Adjusted-R²   : 0.9842
    F-stat (model): 62162.5215   p=0.0000
    MSE           : 2.4657
    RMSE          : 1.5703

clinical_trial (drug_conc)
    drug_concentration → health_improvement
    β coefficient : 0.7049   p=0.0000
    Std Error     : 0.0021
    t-statistic   : 337.1184
    R²            : 0.9913
    Adjusted-R²   : 0.9913
    F-stat (model): 113648.8224   p=0.0000
    MSE           : 1.3584
    RMSE          : 1.1655

ecommerce (visits)
    visits → purchases
    β coefficient : 1.3